# Task 4 – Data Storytelling & Statistical Validation
## Customer Churn Analysis
**ApexPlanet Software Pvt. Ltd.** | **Rahul Kumar Ojha**

### Objective
Synthesize Tasks 1–3 into a compelling business narrative and use basic statistical methods to validate key findings.

### Primary Business Hypothesis
**H0:** Mean Monthly Charges are equal for churned and retained customers.  
**H1:** Mean Monthly Charges are different.

**Test:** Welch's independent two-sample t-test | **α = 0.05**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

df = pd.read_csv("Customer_Churn_Cleaned(2).csv")
df["ChurnFlag"] = df["Churn"].map({"Yes":1,"No":0})
df.head()


## 1. Executive KPIs

In [ ]:
kpis = pd.DataFrame({
    "Metric":["Total Customers","Churned Customers","Churn Rate (%)",
              "Average Monthly Charges","Average Tenure (Months)"],
    "Value":[len(df), int(df.ChurnFlag.sum()), round(df.ChurnFlag.mean()*100,2),
             round(df.MonthlyCharges.mean(),2), round(df.tenure.mean(),2)]
})
kpis


## 2. Business Story from Tasks 1–3

In [ ]:
contract_churn = df.groupby("Contract")["ChurnFlag"].mean().mul(100).sort_values(ascending=False)
contract_churn.to_frame("Churn Rate (%)")


In [ ]:
print("Overall churn rate:", round(df.ChurnFlag.mean()*100,2), "%")
print("Highest churn contract:", contract_churn.index[0], "-", round(contract_churn.iloc[0],2), "%")
print("Churned average monthly charges:", round(df.loc[df.Churn=="Yes","MonthlyCharges"].mean(),2))
print("Retained average monthly charges:", round(df.loc[df.Churn=="No","MonthlyCharges"].mean(),2))


## 3. Primary Hypothesis Test – Welch T-Test

In [ ]:
churned = df.loc[df.Churn=="Yes","MonthlyCharges"].dropna()
retained = df.loc[df.Churn=="No","MonthlyCharges"].dropna()

t_stat, p_value = stats.ttest_ind(churned, retained, equal_var=False)
print("Churned mean:", round(churned.mean(),2))
print("Retained mean:", round(retained.mean(),2))
print("t-statistic:", round(t_stat,4))
print("p-value:", p_value)


In [ ]:
# 95% confidence interval for Churned - Retained mean difference
n1,n2=len(churned),len(retained)
m1,m2=churned.mean(),retained.mean()
s1,s2=churned.std(ddof=1),retained.std(ddof=1)
difference=m1-m2
se=np.sqrt(s1**2/n1+s2**2/n2)
welch_df=(s1**2/n1+s2**2/n2)**2/((s1**2/n1)**2/(n1-1)+(s2**2/n2)**2/(n2-1))
critical=stats.t.ppf(.975,welch_df)
ci_low=difference-critical*se
ci_high=difference+critical*se
print("Mean difference:", round(difference,2))
print("95% CI:", (round(ci_low,2),round(ci_high,2)))
print("Welch degrees of freedom:", round(welch_df,2))


In [ ]:
alpha=0.05
decision = "Reject H0" if p_value < alpha else "Fail to reject H0"
print("Decision:", decision)
print("Business conclusion:", 
      "Monthly Charges differ significantly between churned and retained customers."
      if p_value < alpha else
      "There is insufficient evidence of a difference in Monthly Charges.")


## 4. Independent Validation – Chi-Square Test

**H0:** Churn is independent of Contract Type.  
**H1:** Churn is associated with Contract Type.  
**α = 0.05**


In [ ]:
contract_table=pd.crosstab(df["Contract"],df["Churn"])
chi2,p_chi,chi_df,expected=stats.chi2_contingency(contract_table)
display(contract_table)
print("Chi-square:", round(chi2,4))
print("p-value:", p_chi)
print("Degrees of freedom:", chi_df)
print("Decision:", "Reject H0" if p_chi<0.05 else "Fail to reject H0")


## 5. Visual Evidence

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df,x="Churn",y="MonthlyCharges")
plt.title("Monthly Charges by Churn Status")
plt.xlabel("Churn")
plt.ylabel("Monthly Charges")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9,5))
sns.barplot(x=contract_churn.index,y=contract_churn.values)
plt.title("Churn Rate by Contract Type")
plt.xlabel("Contract")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 6. Business Recommendations

1. Prioritize retention for high-churn contract groups.
2. Review pricing and service experience for customers with higher monthly charges.
3. Strengthen early-tenure onboarding and proactive support.
4. Monitor churn KPIs and customer segments in Power BI.
5. Test retention interventions with controlled experiments.

**Important:** Statistical association or difference does not prove causation.


## 7. Stakeholder Call to Action

**IDENTIFY → VALIDATE → PRIORITIZE → INTERVENE → MEASURE**

Use evidence from the analysis to identify risk, validate important patterns, prioritize customers, intervene, and measure outcomes.


## 8. Final Conclusion

Tasks 1–3 established the data foundation, EDA findings, KPIs and customer-risk segmentation. Task 4 combines these findings into a stakeholder-ready business story and adds statistical validation using Welch's t-test and chi-square testing.